# AWS Bedrock Lambda Integration

This notebook demonstrates how to integrate Amazon Bedrock foundation models with AWS Lambda functions through the tool calling capability. This pattern enables AI applications to perform actions beyond text generation by invoking external services and APIs.

## Learning Objectives

By the end of this notebook, you will understand:
- How to define and configure tools for Bedrock models
- The complete workflow for tool calling and result integration
- Best practices for production Lambda integration
- Error handling and conversation management patterns

## Architecture Overview

The implementation follows this flow:
1. **User Request**: Natural language input requiring computation
2. **Model Analysis**: Bedrock determines if tools are needed
3. **Tool Invocation**: Lambda function executes the requested operation
4. **Result Integration**: Tool output is incorporated into the response
5. **Natural Response**: Final answer presented to the user

## Service Initialization and Configuration

This section establishes connections to AWS services and configures the foundation model for tool-enabled conversations.

### Key Components:

- **Bedrock Runtime Client**: Handles model invocation and conversation management
- **Lambda Client**: Executes serverless functions for tool operations
- **Nova Micro Model**: AWS foundation model optimized for tool calling scenarios

### Tool Schema Definition

The tool specification defines the interface between the model and external functions. Proper schema design ensures reliable tool invocation and parameter validation.

In [ ]:
# Setup Bedrock and Lambda
import boto3
import json

bedrock = boto3.client(service_name='bedrock-runtime')
lambda_client = boto3.client("lambda")
MODEL_ID = "amazon.nova-micro-v1:0"

# Define the calculation tool
math_tool = {
    "toolSpec": {
        "name": "calculateNumbers",
        "description": "Performs basic arithmetic operations",
        "inputSchema": {
            "json": {
                "type": "object",
                "properties": {
                    "operation": {"type": "string"},
                    "num1": {"type": "number"},
                    "num2": {"type": "number"}
                },
                "required": ["operation", "num1", "num2"]
            }
        }
    }
}

# Lambda Integration Function
# This function demonstrates the integration pattern between Bedrock and Lambda services
def execute_calculation(input_data):
    response = lambda_client.invoke(
        FunctionName="math-function",  
        InvocationType="RequestResponse",
        Payload=json.dumps(input_data)
    )
    response_payload = response["Payload"].read()
    calculation_result = json.loads(response_payload)
    response_body = calculation_result.get("body", "{}")
    return json.loads(response_body) if isinstance(response_body, str) else response_body

# Conversation Management and Tool Execution
# This section demonstrates the complete workflow for tool-enabled conversations

# User's initial message
user_input = {
    "role": "user",
    "content": [{"text": "Please subtract 60 from 100"}]
}

# Define system instructions
system_instructions = [
    {"text": """
    You are a virtual assistant capable of performing basic arithmetic operations: add, subtract, multiply, and divide.
    If the user doesn't specify an operation, ask them for more details.
    """}
]

# First interaction with the model
first_interaction = bedrock.converse(
    modelId=MODEL_ID,
    system=system_instructions,
    messages=[user_input],
    toolConfig={
        "tools": [math_tool],
        "toolChoice": {"auto": {}}
    },
    inferenceConfig={"temperature": 0.7}
)

# Process the assistant's response to check if tool is required
assistant_reply = first_interaction["output"]["message"]
message_parts = assistant_reply["content"]
tool_request_block = next((
    part 
    for part in message_parts 
    if "toolUse" in part
), None)

if not tool_request_block:
    print("=== Assistant's Direct Response ===")
    print(message_parts[0]["text"])
else:
    tool_request = tool_request_block["toolUse"]
    tool_input_data = tool_request["input"]
    tool_id = tool_request["toolUseId"]
    print(tool_request_block)
    print(f"→ Assistant triggered tool: calculateNumbers with input: {tool_input_data}")

    # Execute the requested tool
    tool_result = execute_calculation(tool_input_data)
    print(f"← Lambda Function output: {tool_result}")

    # Create a response based on the tool's output
    try:
        result_summary = f"The outcome of the calculation is {tool_result['result']}."
    except Exception as e:
        result_summary = f"Oops! There was an error with the calculation. ({str(e)})"

    # Generate tool result message
    tool_response_msg = {
        "role": "user",
        "content": [
            {
                "toolResult": {
                    "toolUseId": tool_id,
                    "content": [{"text": result_summary}]
                }
            }
        ]
    }

    # Send tool result back to the model
    final_output = bedrock.converse(
        modelId=MODEL_ID,
        messages=[user_input, assistant_reply, tool_response_msg],
        toolConfig={  
            "tools": [math_tool],
            "toolChoice": {"auto": {}}
        },
        inferenceConfig={"temperature": 0.7}
    )

    # Display the final response from the assistant
    final_message = final_output["output"]["message"]["content"][0]["text"]
    print("\n=== Final Assistant Response ===")
    print(final_message)

{'toolUse': {'toolUseId': 'tooluse_EfZdZNuVQgeFtIC2EAPi3Q', 'name': 'calculateNumbers', 'input': {'num1': 100, 'operation': 'subtract', 'num2': 60}}}
→ Assistant triggered tool: calculateNumbers with input: {'num1': 100, 'operation': 'subtract', 'num2': 60}
← Lambda Function output: {'result': 40.0}

=== Final Assistant Response ===
The outcome of subtracting 60 from 100 is 40.0.


## Tool Calling Workflow Analysis

The above implementation demonstrates a complete tool calling workflow with several critical components:

### 1. Initial Model Invocation

The first call to `bedrock.converse()` includes:
- **System Instructions**: Define the assistant's role and capabilities
- **Tool Configuration**: Specify available tools and selection strategy
- **Auto Tool Choice**: Allows the model to decide when to use tools

### 2. Tool Detection and Execution

The code checks the model's response for tool usage requests:
- Parses the response content for `toolUse` blocks
- Extracts tool parameters and invocation ID
- Executes the Lambda function with provided parameters

### 3. Result Integration

Tool results are fed back to the model:
- Creates a `toolResult` message with the Lambda output
- Maintains conversation context with message history
- Generates a natural language response incorporating the results

### 4. Error Handling Patterns

The implementation includes basic error handling:
- Try-catch blocks for tool execution failures
- Graceful degradation when tools are unavailable
- User-friendly error messages for debugging

## Production Implementation Considerations

When deploying tool-enabled AI applications in production environments, several architectural and operational considerations become critical:

### Security and Access Control

- **IAM Permissions**: Implement least-privilege access for Bedrock and Lambda
- **Tool Validation**: Validate tool inputs before Lambda execution
- **Rate Limiting**: Implement throttling to prevent abuse
- **Audit Logging**: Track tool usage and execution patterns

### Performance Optimization

- **Lambda Cold Starts**: Use provisioned concurrency for critical tools
- **Timeout Management**: Set appropriate timeouts for tool execution
- **Caching Strategies**: Cache frequently used tool results
- **Async Processing**: Consider async patterns for long-running tools

### Scalability Patterns

- **Tool Registry**: Centralized management of available tools
- **Version Control**: Manage tool schema evolution
- **Load Balancing**: Distribute tool execution across regions
- **Circuit Breakers**: Implement fallback mechanisms for tool failures

### Monitoring and Observability

- **CloudWatch Metrics**: Monitor tool usage and performance
- **X-Ray Tracing**: Track request flows across services
- **Error Tracking**: Implement comprehensive error monitoring
- **Cost Optimization**: Monitor and optimize Lambda and Bedrock usage

## Advanced Integration Patterns

This basic calculator example can be extended to support more complex enterprise use cases:

### Database Integration

```python
# Example tool for database queries
database_tool = {
    "toolSpec": {
        "name": "queryDatabase",
        "description": "Execute SQL queries on customer database",
        "inputSchema": {
            "json": {
                "type": "object",
                "properties": {
                    "query": {"type": "string"},
                    "parameters": {"type": "array"}
                }
            }
        }
    }
}
```

### API Integration

```python
# Example tool for external API calls
api_tool = {
    "toolSpec": {
        "name": "callExternalAPI",
        "description": "Make HTTP requests to external services",
        "inputSchema": {
            "json": {
                "type": "object",
                "properties": {
                    "endpoint": {"type": "string"},
                    "method": {"type": "string"},
                    "payload": {"type": "object"}
                }
            }
        }
    }
}
```

### Multi-Tool Workflows

Complex applications often require multiple tool calls in sequence:
- **Data Retrieval**: Query databases or APIs for information
- **Processing**: Perform calculations or transformations
- **Action Execution**: Update systems or send notifications
- **Result Aggregation**: Combine outputs from multiple tools

## Key Learnings and Best Practices

This notebook demonstrates several important concepts for building production-ready AI applications with tool integration:

### Technical Implementation

1. **Tool Schema Design**: Precise schema definition ensures reliable tool invocation
2. **Conversation Management**: Maintaining message history preserves context
3. **Error Handling**: Graceful degradation improves user experience
4. **Service Integration**: Clean separation between AI and business logic

### Architectural Patterns

1. **Serverless Integration**: Lambda provides scalable tool execution
2. **Event-Driven Design**: Tools respond to model requests asynchronously
3. **Microservices Architecture**: Each tool can be independently deployed
4. **API Gateway Pattern**: Centralized tool management and routing

### Production Readiness

1. **Security**: Implement proper authentication and authorization
2. **Monitoring**: Track performance and usage patterns
3. **Scalability**: Design for high-volume tool execution
4. **Reliability**: Implement retry logic and circuit breakers

### Next Steps

To extend this implementation:
- Add more complex tools for real business use cases
- Implement tool chaining for multi-step workflows
- Add comprehensive error handling and logging
- Integrate with enterprise systems and databases
- Implement tool versioning and deployment strategies